# 01 · Transcription (Whisper large-v3)

First notebook of the Grammar Scoring Engine pipeline. It transcribes all 985 clips (769 train, 216 test) and saves, for every clip, the transcript, word-level timestamps and Whisper's own quality signals. Later notebooks read these from the saved file, so the slow GPU step runs only once.

**Why verbatim transcripts.** Grammar is judged on what the speaker actually said, but Whisper tends to "clean up" speech (dropping fillers, repetitions and false starts). The decoding prompt imitates disfluent speech so that Whisper keeps them; the prompt itself contains no grammar errors, so it cannot push Whisper to invent any.

**Outputs** (in `/kaggle/working`)
- `transcripts_whisper_large_v3_prompted.parquet`: the file used downstream (exact types; empty transcripts stay empty)
- `transcripts_whisper_large_v3_prompted.csv`: human-readable copy
- `transcripts_whisper_large_v3_prompted_config.json`: model and decoding settings
- `transcripts_whisper_large_v3_prompted.jsonl`: per-file checkpoint written during the run

**Settings:** Accelerator GPU T4 x2, Internet on. Runtime about 1 h 45 min.

## 1. Configuration
All decoding settings in one place:
- **Fallback temperatures** and the **compression-ratio / log-probability thresholds**: a 30-second window whose output is repetitive or low-confidence is decoded again with a little randomness. This prevents repetition loops ("and, and, and, …") on trailing silence.
- **`condition_on_previous_text`** carries the verbatim style of the prompt into the second 30-second window (clips are 45–60 s).
- **`hallucination_silence_threshold`** skips long silences in which Whisper produced text that was not spoken.
- **`vad_filter=False`** keeps all audio: voice-activity filtering can drop quiet, hesitant speech.
- **`LIMIT`** transcribes only a few files per split for a quick test; outputs are then saved with a `_DEBUG` suffix.

In [2]:
RUN_NAME = "whisper_large_v3_prompted"
MODEL_NAME = "large-v3"

# The prompt nudges Whisper to keep fillers, repetitions and false starts instead of producing a cleaned-up version of the speech.
PROMPT = "Umm, so, uh, I think, I think that the... the main thing is, like, you know, we have to, um, go there."

LIMIT = None

EXPECTED_TRAIN, EXPECTED_TEST = 769, 216
SR = 16000

TRANSCRIBE_KW = dict(
    language="en",
    task="transcribe",
    beam_size=5,
    temperature=(0.0, 0.2, 0.4, 0.6, 0.8, 1.0),  # retry a window with more randomness if its output is bad
    compression_ratio_threshold=2.4,             # "too repetitive" -> triggers a retry
    log_prob_threshold=-1.0,                     # "too low confidence" -> triggers a retry
    no_speech_threshold=0.6,
    initial_prompt=PROMPT,
    condition_on_previous_text=True,
    word_timestamps=True,
    hallucination_silence_threshold=2.0,         # skip silences over 2s where hallucinated text is detected
    vad_filter=False,
)

## 2. Data and file matching
Locate the competition data, inspect the CSVs, and match every CSV row to its audio file. The columns are detected from the data rather than assumed. The checks stop the notebook if a row has no audio file, a name is duplicated, or the row counts differ from the competition description. Train and test share 212 file names (e.g. `audio_1` exists in both folders), so every file is identified by **split + file name**.

In [3]:
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")
train_csv_candidates = sorted(INPUT_ROOT.rglob("train.csv"))
assert len(train_csv_candidates) == 1, f"Expected exactly one train.csv, found: {train_csv_candidates}"

DATA_DIR = train_csv_candidates[0].parent
print("DATA_DIR:", DATA_DIR)
for p in sorted(DATA_DIR.iterdir()):
    print("  ", p.name)

DATA_DIR: /kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final
   sample_submission.csv
   test
   test.csv
   train
   train.csv


In [4]:
import pandas as pd

train_raw = pd.read_csv(DATA_DIR / "train.csv")
test_raw = pd.read_csv(DATA_DIR / "test.csv")
sub_raw = pd.read_csv(DATA_DIR / "sample_submission.csv")  # format reference only

for name, df in [("train.csv", train_raw), ("test.csv", test_raw), ("sample_submission.csv", sub_raw)]:
    print(f"--- {name}: {df.shape}")
    print(df.dtypes.to_string())
    display(df.head(3))

--- train.csv: (769, 2)
filename     object
label       float64


,filename,label
0,audio_192.wav,3.0
1,audio_436.wav,3.0
2,audio_447.wav,3.5


--- test.csv: (216, 2)
filename    object
label        int64


,filename,label
0,audio_128.wav,-1
1,audio_156.wav,-1
2,audio_19.wav,-1


--- sample_submission.csv: (204, 2)
filename    object
label        int64


,filename,label
0,audio_804.wav,-1
1,audio_1028.wav,-1
2,audio_865.wav,-1


In [5]:
def index_audio(folder: Path) -> dict:
    """Map file name -> full path"""
    files = [p for p in folder.rglob("*") if p.suffix.lower() == ".wav"]
    index = {p.stem: p for p in files}
    assert len(index) == len(files), f"Two files share a name in {folder}"
    return index

def detect_file_col(df: pd.DataFrame, audio_index: dict) -> str:
    """Find the column whose values match the audio file names."""
    for col in df.columns:
        stems = df[col].astype(str).map(lambda s: Path(s).stem)
        if stems.isin(audio_index.keys()).mean() > 0.9:
            return col
    raise ValueError(f"No column matches audio file names. Columns: {list(df.columns)}")


def build_manifest(df: pd.DataFrame, audio_index: dict, split: str) -> pd.DataFrame:
    """One row per audio file: id, original CSV name, label, split, path."""
    file_col = detect_file_col(df, audio_index)
    numeric_cols = [c for c in df.columns if c != file_col and pd.api.types.is_numeric_dtype(df[c])]
    assert len(numeric_cols) == 1, f"Expected one label column in {split}, found {numeric_cols}"

    out = pd.DataFrame({
        "file_id": df[file_col].astype(str).map(lambda s: Path(s).stem),
        "orig_name": df[file_col].astype(str),  # exact value from the CSV, reused for the submission
        "label": df[numeric_cols[0]].astype(float),
        "split": split,
    })
    out["path"] = out["file_id"].map(audio_index)

    missing = out.loc[out["path"].isna(), "orig_name"].tolist()
    assert not missing, f"{len(missing)} {split} rows have no audio file, e.g. {missing[:5]}"
    assert out["file_id"].is_unique, f"Duplicate file names in {split}.csv"

    extra = set(audio_index) - set(out["file_id"])
    print(f"{split}: file column='{file_col}', label column='{numeric_cols[0]}', "
          f"{len(out)} rows, {len(extra)} audio files not listed in the CSV")
    return out


train_audio = index_audio(DATA_DIR / "train")
test_audio = index_audio(DATA_DIR / "test")

train_m = build_manifest(train_raw, train_audio, "train")
test_m = build_manifest(test_raw, test_audio, "test")

assert len(train_m) == EXPECTED_TRAIN, f"train rows: {len(train_m)} != {EXPECTED_TRAIN}"
assert len(test_m) == EXPECTED_TEST, f"test rows: {len(test_m)} != {EXPECTED_TEST}"
assert (test_m["label"] == -1).all(), "Test labels are not all -1; check which column is the label"


shared = set(train_m["file_id"]) & set(test_m["file_id"])
print(f"File names present in both train and test: {len(shared)} (handled by keying on split + file_id)")

print("\nTrain label distribution:")
print(train_m["label"].value_counts().sort_index().to_string())

manifest = pd.concat([train_m, test_m], ignore_index=True)
if LIMIT is not None:
    manifest = manifest.groupby("split", group_keys=False).head(LIMIT).reset_index(drop=True)
    print(f"\n DEBUG MODE: only {LIMIT} files per split ")
print(f"\nFiles to transcribe: {len(manifest)}")

train: file column='filename', label column='label', 769 rows, 0 audio files not listed in the CSV
test: file column='filename', label column='label', 216 rows, 0 audio files not listed in the CSV
File names present in both train and test: 212 (handled by keying on split + file_id)

Train label distribution:
label
0.0     37
1.0      1
1.5      3
2.0    102
2.5     79
3.0    174
3.5     64
4.0    124
4.5     52
5.0    133

Files to transcribe: 985


## 3. Data checks
- **Duplicates:** no two files have identical audio content, so the shared names are a naming coincidence, not leakage between train and test.
- **Label 0** is not defined by the rubric (which runs 1–5). Label-0 clips are not short or truncated (mean 57.9 s), so the label most likely marks recordings that could not be scored.
- **Clip length differs between train and test:** train is mostly ~60 s clips, test mostly ~45 s clips, and ~45 s clips have a lower and narrower score distribution in train. This is why every downstream feature is a rate or proportion rather than a raw count, and why validation results are also reported separately for short clips.

In [6]:
import hashlib
import soundfile as sf


def file_md5(path, chunk_size=1 << 20) -> str:
    """Fingerprint a file's exact bytes; identical files give identical fingerprints."""
    h = hashlib.md5()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk_size), b""):
            h.update(block)
    return h.hexdigest()


# Are any audio files duplicated (same content), within or across splits?
manifest["md5"] = [file_md5(p) for p in manifest["path"]]
dup_groups = manifest[manifest.duplicated("md5", keep=False)].sort_values("md5")
print(f"Files whose audio content appears more than once: {len(dup_groups)}")

shared_ids = set(train_m["file_id"]) & set(test_m["file_id"])
cross = manifest[manifest["file_id"].isin(shared_ids)].pivot(index="file_id", columns="split", values="md5")
print(f"Shared names with identical train/test audio: {(cross['train'] == cross['test']).sum()} / {len(cross)}")
if not dup_groups.empty:
    display(dup_groups[["split", "file_id", "label", "md5"]].head(20))

# Durations by label: are label-0 clips simply short or truncated recordings?
manifest["duration_sec"] = [sf.info(str(p)).duration for p in manifest["path"]]
print("\nDuration by train label:")
print(manifest[manifest["split"] == "train"].groupby("label")["duration_sec"]
      .describe()[["count", "mean", "min", "max"]].round(1).to_string())
print("\nTest durations:")
print(manifest.loc[manifest["split"] == "test", "duration_sec"].describe().round(1).to_string())

manifest = manifest.drop(columns=["md5", "duration_sec"])  # keep the manifest unchanged for later cells

In [7]:
manifest["duration_sec"] = [sf.info(str(p)).duration for p in manifest["path"]]

for split in ["train", "test"]:
    d = manifest.loc[manifest["split"] == split, "duration_sec"]
    print(f"{split}: ~45s: {d.between(44.5, 45.5).sum()} | ~60s: {d.between(59.5, 61.1).sum()} "
          f"| under 30s: {(d < 30).sum()} | total: {len(d)}")

short_test = manifest[(manifest["split"] == "test") & (manifest["duration_sec"] < 30)]
display(short_test[["file_id", "duration_sec"]])

manifest = manifest.drop(columns="duration_sec")  # keep the manifest unchanged for later cells

train: ~45s: 114 | ~60s: 549 | under 30s: 14 | total: 769
test: ~45s: 115 | ~60s: 58 | under 30s: 3 | total: 216


,file_id,duration_sec
782,audio_28,18.560000
865,audio_107,6.480000
966,audio_48,28.266687


In [8]:
manifest["duration_sec"] = [sf.info(str(p)).duration for p in manifest["path"]]
manifest["dur_group"] = pd.cut(manifest["duration_sec"], bins=[0, 30, 50, 62],
                               labels=["<30s", "~45s", "~60s"])

tr = manifest[manifest["split"] == "train"]
print("Train label distribution by duration group:")
print(pd.crosstab(tr["dur_group"], tr["label"]).to_string())
print("\nMean label by group:")
print(tr.groupby("dur_group", observed=True)["label"].agg(["count", "mean", "std"]).round(2).to_string())

manifest = manifest.drop(columns=["duration_sec", "dur_group"])  # keep the manifest unchanged for later cells

Train label distribution by duration group:
label      0.0  1.0  1.5  2.0  2.5  3.0  3.5  4.0  4.5  5.0
dur_group                                                  
<30s         1    0    0    2    1    2    0    8    0    0
~45s         3    1    0   37   22   42   14   35    4   10
~60s        33    0    3   63   56  130   50   81   48  123

Mean label by group:
           count  mean   std
dur_group                   
<30s          14  3.18  1.20
~45s         168  3.05  0.98
~60s         587  3.39  1.30


## 4. Load Whisper
`faster-whisper` runs the same Whisper models several times faster than the reference implementation. Its engine (CTranslate2) sometimes cannot find the CUDA libraries that ship with PyTorch, so they are preloaded first. The model runs in float16 on a T4 GPU, with a float32 fallback for GPUs that do not support it.

In [10]:
!pip install -q faster-whisper

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 60.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.6/39.6 MB 54.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 75.1 MB/s eta 0:00:00:00:0100:01


In [11]:
import ctypes
import glob
import os
import site


def preload_nvidia_libs():
    """Load the CUDA math libraries that ship with PyTorch so CTranslate2 can find them"""
    lib_dirs = []
    for sp in site.getsitepackages():
        for sub in ("cublas", "cudnn"):
            d = os.path.join(sp, "nvidia", sub, "lib")
            if os.path.isdir(d):
                lib_dirs.append(d)
    pending = sorted({f for d in lib_dirs for f in glob.glob(os.path.join(d, "lib*.so*"))})
    loaded = []
    for _ in range(5):  # retry because some libraries depend on others loading first
        failed = []
        for f in pending:
            try:
                ctypes.CDLL(f, mode=ctypes.RTLD_GLOBAL)
                loaded.append(f)
            except OSError:
                failed.append(f)
        if len(failed) == len(pending):
            break
        pending = failed
    return loaded, pending


loaded, not_loaded = preload_nvidia_libs()
print(f"Preloaded {len(loaded)} NVIDIA libraries ({len(not_loaded)} skipped)")

import ctranslate2
ctranslate2.set_random_seed(42)
from faster_whisper import WhisperModel

assert ctranslate2.get_cuda_device_count() > 0, \
    "No GPU visible. Set Accelerator to 'GPU T4 x2' in the side panel and restart the session."

supported = ctranslate2.get_supported_compute_types("cuda")
COMPUTE_TYPE = "float16" if "float16" in supported else "float32"
print("Compute type:", COMPUTE_TYPE)

model = WhisperModel(MODEL_NAME, device="cuda", device_index=0, compute_type=COMPUTE_TYPE)
print("Model loaded:", MODEL_NAME)

Preloaded 11 NVIDIA libraries (0 skipped)
Compute type: float16


Model loaded: large-v3


## 5. Transcription function
Audio is read with `soundfile` and converted to 16 kHz mono (`faster-whisper`'s own audio decoder is incompatible with the installed PyAV version). For each clip the function returns the transcript, every word with its start time, end time and confidence, and three quality signals: mean log-probability (confidence), maximum no-speech probability, and maximum compression ratio (values above 2.4 indicate repetitive output). Transcription is lazy in `faster-whisper`, so the segments are consumed explicitly. The loader is checked on one file before any GPU time is spent.

In [12]:
import json
from math import gcd
from statistics import mean

import numpy as np
import soundfile as sf
from scipy.signal import resample_poly


def load_audio_16k(path) -> np.ndarray:
    """Read a WAV file and return 16 kHz mono float32 audio"""
    audio, sr = sf.read(str(path), dtype="float32", always_2d=True)  # shape: (samples, channels)
    audio = audio.mean(axis=1)                                       # stereo -> mono
    if sr != SR:
        g = gcd(sr, SR)
        audio = resample_poly(audio, SR // g, sr // g)               # high-quality resampling
    return np.ascontiguousarray(audio, dtype=np.float32)


def transcribe_file(path, kw=None) -> dict:
    """Transcribe one audio file and return the transcript, word timings and quality signals."""
    kw = TRANSCRIBE_KW if kw is None else kw
    audio = load_audio_16k(path)
    duration = len(audio) / SR

    segments, _info = model.transcribe(audio, **kw)
    segments = list(segments)  # REQUIRED: transcription is lazy and only runs when consumed

    words = [
        [w.word.strip(), round(float(w.start), 3), round(float(w.end), 3), round(float(w.probability), 4)]
        for s in segments for w in (s.words or [])
    ]
    text = " ".join(s.text.strip() for s in segments).strip()

    return {
        "duration_sec": round(duration, 3),
        "transcript": text,
        "n_words": len(words),
        "n_segments": len(segments),
        "words_json": json.dumps(words),  # [word, start_sec, end_sec, probability]
        "mean_avg_logprob": round(mean(float(s.avg_logprob) for s in segments), 4) if segments else None,
        "max_no_speech_prob": round(max(float(s.no_speech_prob) for s in segments), 4) if segments else None,
        "max_compression_ratio": round(max(float(s.compression_ratio) for s in segments), 4) if segments else None,
    }


# Check the loader on one file before any transcription:
# the loaded length must match the duration in the file header.
_p = train_m["path"].iloc[0]
_a = load_audio_16k(_p)
_info = sf.info(str(_p))
print(f"Native sample rate: {_info.samplerate} Hz, channels: {_info.channels}")
print(f"Loaded: {len(_a)} samples, dtype {_a.dtype}, {len(_a) / SR:.2f}s (header says {_info.duration:.2f}s)")
assert abs(len(_a) / SR - _info.duration) < 0.05, "Loaded duration doesn't match the file header"
assert np.isfinite(_a).all(), "Audio contains NaN or infinite values"

Native sample rate: 16000 Hz, channels: 1
Loaded: 968134 samples, dtype float32, 60.51s (header says 60.51s)


## 6. Smoke test
Three training clips (lowest, middle and highest score) are transcribed before the full run. During development, these transcripts were checked by listening to the audio: fillers and repetitions are kept, and no prompt text leaks into the output.

In [14]:
by_label = train_m.sort_values("label").reset_index(drop=True)
smoke = by_label.iloc[[0, len(by_label) // 2, len(by_label) - 1]]

for row in smoke.itertuples(index=False):
    r = transcribe_file(row.path)
    print(f"\n=== {row.file_id} | label={row.label} | {r['duration_sec']}s | {r['n_words']} words "
          f"| logprob={r['mean_avg_logprob']} | compression={r['max_compression_ratio']}")
    print(r["transcript"])


=== audio_5073 | label=0.0 | 60.075s | 127 words | logprob=-0.3228 | compression=1.8802
I'm going to tell you about the scene at the product market. The product market is full of people and selling and buying these vegetables, etc. In the market, you can find vegetables. People do the work on this, selling and buying. The vegetables will be cost on low or high. But the sounds which they say that these are the vegetables and that these are the vegetables that are available, these are the sounds which can be heard in the market. And those sounds are even audible in the road also. The product market changes day by day from morning to evening. In one day morning to evening, the crowd changes and the teams, the parties, the bar teams, those are the teams that will change.



=== audio_489 | label=3.0 | 60.075s | 121 words | logprob=-0.234 | compression=1.8855
The scene of a crowded market is something like, like a very fish market, and it is all about the selling vegetables on the market, and so many people are coming on a particular day where there is more selling of vegetables, and on the morning time, it will be very crowded in the market that, uh, many people were coming to buy the vegetables, and the sellers are shouting loudly to buy, to sell the vegetables to the customers, and the customers were also bargaining to the sellers to get the vegetables with the low price, and, and the scene of the crowded market will be, it was something rushy, and we can, it was something rushy, and we can,



=== audio_592 | label=5.0 | 60.075s | 112 words | logprob=-0.2737 | compression=1.688
The journey to Switzerland is an adventure in itself, whether you arrive by plane, train, or car. Now, if you fly into Zurich or Genoa, you'll be greeted by stunning views of the Swiss Alps as your plane descends towards the airport. From there, you can take a scenic train ride to your destination, passing through picturesque villages and rolling countryside, alternatively driving through Switzerland offers the flexibility to stop at charming towns along the way and take in breathtaking landscapes at your own pace. Whichever mode of transportation you choose, the journey to Switzerland is filled with anticipation and excitement as you anticipate the beauty that awaits. SwissCuisine is a renown...


## 7. Full run (checkpointed, with fallbacks)
Every train and test file is transcribed with **identical settings**, so the downstream models see the same kind of transcript at training and test time. Each result is appended to a JSONL checkpoint as soon as it is finished; rerunning the cell resumes where it stopped.

For two files (`audio_373`, `audio_709`), faster-whisper's word-alignment step crashed with an `IndexError` inside the library. Such files are retried automatically with fallback settings, and the setting used is recorded:
1. `no_prev_text`: without conditioning on previous text (different segmentation; word timings kept). `audio_373` succeeded here.
2. `no_word_timestamps`: transcript only. `audio_709` succeeded here; its fluency features (which need word timings) are therefore missing and handled as unknown downstream.

*Note: in the original run these retries were carried out in separate cells after the main loop; they are folded into the loop here so the notebook runs top to bottom. The procedure is the same.*

In [15]:
from tqdm.auto import tqdm

OUT_DIR = Path("/kaggle/working")
suffix = "" if LIMIT is None else "_DEBUG"
RAW_PATH = OUT_DIR / f"transcripts_{RUN_NAME}{suffix}.jsonl"

# Fallback settings for files where faster-whisper's word alignment fails (an IndexError inside the library).
# They are tried in order only after the default settings fail; each record notes which settings were used.
_base_kw = {k: v for k, v in TRANSCRIBE_KW.items() if k != "hallucination_silence_threshold"}
FALLBACKS = [
    ("no_prev_text", _base_kw | {"condition_on_previous_text": False}),  # different segmentation, keeps word timings
    ("no_word_timestamps", _base_kw | {"word_timestamps": False}),      # last resort: transcript only, no word timings
]


def transcribe_with_fallbacks(path) -> dict:
    errors = []
    for name, kw in [("default", None)] + FALLBACKS:
        try:
            return {**transcribe_file(path, kw), "fallback_settings": None if name == "default" else name}
        except Exception as e:
            errors.append(f"{name}: {type(e).__name__}: {e}")
    raise RuntimeError(" | ".join(errors))


# Resume: skip files already transcribed successfully in this session.
done = set()
if RAW_PATH.exists():
    with open(RAW_PATH) as f:
        for line in f:
            rec = json.loads(line)
            if rec.get("error") is None:
                done.add((rec["split"], rec["file_id"]))
print(f"Already done: {len(done)}")

todo = manifest[[(s, f) not in done for s, f in zip(manifest["split"], manifest["file_id"])]]
print(f"To transcribe now: {len(todo)}")

with open(RAW_PATH, "a") as f:
    for row in tqdm(todo.itertuples(index=False), total=len(todo)):
        rec = {"split": row.split, "file_id": row.file_id}
        try:
            rec.update(transcribe_with_fallbacks(row.path))
            rec["error"] = None
        except Exception as e:
            rec["error"] = f"{type(e).__name__}: {e}"
        f.write(json.dumps(rec) + "\n")
        f.flush()  

Already done: 0
To transcribe now: 985


  0%|          | 0/985 [00:00<?, ?it/s]

## 8. Verification
Before saving: every file must have exactly one successful transcript. The report covers durations, empty transcripts, possible repetition loops (compression ratio > 2.4), prompt leakage, speaking rate by split, and files transcribed with fallback settings. As an end-to-end sanity check, speaking rate should correlate positively with the grammar label (more proficient speakers tend to speak more fluently); a value near zero would point to transcripts paired with the wrong labels.

In [21]:
from scipy.stats import spearmanr

with open(RAW_PATH) as f:
    raw = pd.DataFrame([json.loads(line) for line in f])  # avoids pandas guessing types for names like "123"
raw = raw.drop_duplicates(["split", "file_id"], keep="last")  # retried files keep their latest result

errors = raw[raw["error"].notna()]
if not errors.empty:
    display(errors[["split", "file_id", "error"]])
assert errors.empty, f"{len(errors)} files failed with every setting; inspect the errors above"

result = manifest.merge(raw.drop(columns="error"), on=["split", "file_id"],
                        how="left", validate="one_to_one")
assert len(result) == len(manifest)
assert result["transcript"].notna().all(), "Some files have no transcript; rerun the transcription cell"

result["words_per_min"] = result["n_words"] / (result["duration_sec"] / 60)

used_fallback = result[result["fallback_settings"].notna()] if "fallback_settings" in result else result.iloc[:0]
print(f"Files transcribed with fallback settings: {len(used_fallback)}")
if len(used_fallback):
    display(used_fallback[["split", "file_id", "fallback_settings", "n_words"]])

print("Duration (sec):")
print(result["duration_sec"].describe().round(1).to_string())
print(f"\nEmpty transcripts: {(result['transcript'].str.len() == 0).sum()}")
print(f"Possible repetition loops (compression > 2.4): {(result['max_compression_ratio'] > 2.4).sum()}")

prompt_start = PROMPT[:25].lower()
leaks = result["transcript"].str.lower().str.contains(prompt_start, regex=False)
print(f"Transcripts containing prompt text: {leaks.sum()}")

print("\nWords per minute by split:")
print(result.groupby("split")["words_per_min"].describe().round(1).to_string())

tr = result[result["split"] == "train"]
rho, p = spearmanr(tr["words_per_min"], tr["label"])
print(f"\nSanity check: Spearman(words per minute, label) on train = {rho:.3f} (p={p:.1e})")

flagged = result[(result["max_compression_ratio"] > 2.4) | leaks | (result["n_words"] < 20)]
print(f"\nFiles to inspect manually: {len(flagged)}")
display(flagged[["split", "file_id", "label", "duration_sec", "n_words", "max_compression_ratio", "transcript"]].head(20))

Duration (sec):
count    985.0
mean      54.2
std        8.5
min        6.5
25%       45.1
50%       60.1
75%       60.1
max       61.0

Empty transcripts: 0
Possible repetition loops (compression > 2.4): 0
Transcripts containing prompt text: 0

Words per minute by split:
       count   mean   std   min    25%    50%    75%    max
split                                                      
test   216.0  127.2  24.8  42.6  114.1  127.6  142.8  187.5
train  769.0  116.1  29.2   0.0   98.9  118.9  135.8  210.4

Sanity check: Spearman(words per minute, label) on train = 0.210 (p=3.9e-09)

Files to inspect manually: 4


,split,file_id,label,duration_sec,n_words,max_compression_ratio,transcript
41,train,audio_5069,0.0,60.075,4.0,0.5556,Thank you. Thank you.
51,train,audio_5039,0.0,60.075,6.0,0.5556,Thank you. Thank you. Thank you.
511,train,audio_709,4.0,60.011,0.0,1.5363,My goal in life is to be successful and have l...
865,test,audio_107,-1.0,6.480,8.0,0.8947,My favorite character is Naruto in Captain Ame...


## 9. Save
The Parquet file is the one used by the later notebooks. It is reloaded and compared with what was written before the notebook finishes.

In [22]:
if LIMIT is None:
    assert (result["split"] == "train").sum() == EXPECTED_TRAIN
    assert (result["split"] == "test").sum() == EXPECTED_TEST
else:
    print("!!! DEBUG run: outputs are saved with a _DEBUG suffix and are NOT complete !!!")

result["path"] = result["path"].astype(str)
base = OUT_DIR / f"transcripts_{RUN_NAME}{suffix}"
result.to_parquet(f"{base}.parquet", index=False)  # exact types; empty strings stay empty
result.to_csv(f"{base}.csv", index=False)          # human-readable copy

with open(f"{base}_config.json", "w") as f:
    json.dump({"run_name": RUN_NAME, "model": MODEL_NAME, "compute_type": COMPUTE_TYPE,
               "transcribe_kwargs": TRANSCRIBE_KW, "limit": LIMIT}, f, indent=2)

# Round-trip check: the saved Parquet must load back identical to what was saved.
check = pd.read_parquet(f"{base}.parquet")
assert len(check) == len(result)
assert (check["transcript"] == result["transcript"]).all()
assert (check["file_id"] == result["file_id"]).all()

print("Saved:")
for p in sorted(OUT_DIR.glob(f"transcripts_{RUN_NAME}{suffix}*")):
    print(f"  {p.name}  ({p.stat().st_size / 1e6:.2f} MB)")

Saved:
  transcripts_whisper_large_v3_prompted.csv  (4.24 MB)
  transcripts_whisper_large_v3_prompted.jsonl  (4.29 MB)
  transcripts_whisper_large_v3_prompted.parquet  (1.94 MB)
  transcripts_whisper_large_v3_prompted_config.json  (0.00 MB)
